# Data Collection and Basic Processing Notebook

**Contents**
1. Solar Power Generation Trends (Monthly) NITI 2015 - 2026
    - Extraction
    - Processing
        - Saving as flat file
2. Solar Power Installed Capacity Trends (Yearly) NITI 2015 - 2026
    - Extraction
    - Processing
        - Saving as flat file
3. Peak Demand Trends (Monthly) CEA 2015 - 2026
    - CEA Monthly Peak Demand Report Scraper and CEA PDF Extraction pipeline
    - Processing
        - Saving as flat file
4. Weather Data from NASA POWER API (2015-2026)
    - Unified Extraction and Processing pipline
    - Saving as flat file

Importing relevant libraries

In [40]:
import os
import sys
import requests
import pdfplumber
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select


__________________________

## Solar Power Generation Trends (Monthly) NITI 2015 - 2026

This script retrieves monthly solar electricity generation data from the NITI Aayog ICeD API and stores the raw dataset locally as a CSV file. Existing files are detected and skipped to avoid unnecessary API requests.


### Extraction

In [41]:
# URL to obtain the generation trends data & other relevant params
GENTRENDS_URL = 'https://icedapi.niti.gov.in/v1/gen-metatable-data'

GENTRENDS_HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-GB,en;q=0.9",
    "content-type": "application/json",
    "origin": "https://iced.niti.gov.in",
    "priority": "u=1, i",
    "referer": "https://iced.niti.gov.in/",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"
}

GENTRENDS_PAYLOAD = {
    "source": "solar",
    "rangeType": "monthly"
}


# Path to store the raw csv
GENTRENDS_FILEPATH = "../data/raw/niti_aayog/SOLAR-GENERATION-TRENDS-2015-2026.csv"


In [42]:
# Only trigger API call if local file doesn't exist
if not os.path.isfile(GENTRENDS_FILEPATH):

    print(f'Fetching generation dataset from NITI')

    try:
        response = requests.post(
            url=GENTRENDS_URL, 
            headers=GENTRENDS_HEADERS, 
            json=GENTRENDS_PAYLOAD
        )
        response.raise_for_status()
        print(f'Status code: {response.status_code}')

        if response.status_code == 200:
            gen_data = response.json()

        #Checking if the fetched data is a list
        if isinstance(gen_data, list):
            gen_df = pd.DataFrame(gen_data)
            gen_df.to_csv(GENTRENDS_FILEPATH, index=False)
    except Exception as e:
        print(f'Could not fetch API: {e}')
        
else: 
    gen_df = pd.read_csv(GENTRENDS_FILEPATH)
    print(f'Dataset already exists at {GENTRENDS_FILEPATH}. Skipping API request...')

gen_df

Fetching generation dataset from NITI
Status code: 200


,source,state,generation,year,month
0,solar,Andaman and Nicobar Islands,1.17,2015-16,April
1,solar,Andaman and Nicobar Islands,0.83,2015-16,August
2,solar,Andaman and Nicobar Islands,0.84,2015-16,December
3,solar,Andaman and Nicobar Islands,1.37,2015-16,February
4,solar,Andaman and Nicobar Islands,1.35,2015-16,January
...,...,...,...,...,...
5029,solar,West Bengal,34.11,2026-27,August
5030,solar,West Bengal,30.22,2026-27,July
5031,solar,West Bengal,40.24,2026-27,June
5032,solar,West Bengal,47.38,2026-27,May


### Processing

In [37]:
# Adding a month_name column
MONTH_NAME_TO_NUM = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

gen_df['month_num'] = gen_df['month'].map(MONTH_NAME_TO_NUM)

# Renaming columns to standardize format
gen_df = gen_df.rename(columns={'month': 'month_name', 'year': 'fyear', 'generation': 'generation_mu'})

gen_df = gen_df[['state', 'fyear','month_num', 'month_name', 'generation_mu']]
gen_df

,state,fyear,month_num,month_name,generation_mu
0,Andaman and Nicobar Islands,2015-16,4,April,1.17
1,Andaman and Nicobar Islands,2015-16,8,August,0.83
2,Andaman and Nicobar Islands,2015-16,12,December,0.84
3,Andaman and Nicobar Islands,2015-16,2,February,1.37
4,Andaman and Nicobar Islands,2015-16,1,January,1.35
...,...,...,...,...,...
5029,West Bengal,2026-27,8,August,34.11
5030,West Bengal,2026-27,7,July,30.22
5031,West Bengal,2026-27,6,June,40.24
5032,West Bengal,2026-27,5,May,47.38


#### Saving as a flat file

In [38]:
PROCESSED_GENTRENDS_FILEPATH = '../data/processed/flat_files/SOLAR-GENERATION-TRENDS-2015-2026.csv'
if not os.path.isfile(PROCESSED_GENTRENDS_FILEPATH):
    gen_df.to_csv(PROCESSED_GENTRENDS_FILEPATH, index=False)

__________________________

## Solar Power Installed Capacity Trends (Yearly) NITI 2015 - 2026

This script retrieves installed power generation capacity data from the NITI Aayog ICeD API, filters the dataset to solar energy records, and saves the results as a local CSV file. Existing datasets are skipped to prevent redundant downloads.


### Extraction

In [22]:
IC_TRENDS_URL = "https://icedapi.niti.gov.in/v1/capacity-metatable-data?switchMetaData=false"

IC_TRENDS_HEADERS = {
    "accept" : "application/json, text/plain, */*",
    "accept-language": "en-GB, en; q=0.9",
    "dnt": "1", 
    "origin": "https://iced.niti.gov.in", 
    "referer": "https://iced.niti.gov.in/", 
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"
}

IC_TRENDS_FILEPATH = "../data/raw/niti_aayog/INSTALLED-CAPACITY-TRENDS-2015-2026.csv"

In [23]:
# Only trigger API call if local file doesn't exist
if not os.path.isfile(IC_TRENDS_FILEPATH):
    print(f'Fetching capacity dataset from NITI...')
    try:
        response = requests.get(url=IC_TRENDS_URL, headers=IC_TRENDS_HEADERS)
        response.raise_for_status()
        print(f'Status code: {response.status_code}')
        ic_trends_data = response.json()
        #Checking if the fetched data is a list
        if isinstance(ic_trends_data, list):
            ic_trends_df = pd.DataFrame(ic_trends_data)
        ic_trends_df.to_csv(IC_TRENDS_FILEPATH, index=False)
    except Exception as e:
        print(f'Could not fetch API {e}')

else: 
    print(f'Dataset already exists at {IC_TRENDS_FILEPATH}. Skipping API request...')
    ic_trends_df = pd.read_csv(IC_TRENDS_FILEPATH)

Dataset already exists at ../data/raw/niti_aayog/INSTALLED-CAPACITY-TRENDS-2015-2026.csv. Skipping API request...


### Processing

In [24]:
# Filtering for only solar data
ic_trends_df = ic_trends_df[ic_trends_df['source'].str.strip() =='solar']

#Standardizing feature names
ic_trends_df = ic_trends_df.rename(columns={'capacity': 'capacity_mw'})
ic_trends_df

,state,fyear,source,capacity_mw
0,Andaman and Nicobar Islands,2015-16,solar,5.34
1,Andaman and Nicobar Islands,2016-17,solar,6.80
2,Andaman and Nicobar Islands,2017-18,solar,6.80
3,Andaman and Nicobar Islands,2018-19,solar,11.97
4,Andaman and Nicobar Islands,2019-20,solar,12.43
...,...,...,...,...
433,West Bengal,2022-23,solar,179.98
434,West Bengal,2023-24,solar,194.07
435,West Bengal,2024-25,solar,320.62
436,West Bengal,2025-26,solar,320.62


#### Saving as a flat file

In [25]:
PROCESSED_ICTRENDS_FILEPATH = '../data/processed/flat_files/INSTALLED-CAPACITY-TRENDS-2015-2026.csv'
if not os.path.isfile(PROCESSED_ICTRENDS_FILEPATH):
    ic_trends_df.to_csv(PROCESSED_ICTRENDS_FILEPATH, index=False)

__________________________

## Peak Demand Trends (Monthly) CEA 2015 - 2026

### CEA Monthly Peak Demand Report Scraper and CEA PDF Extraction pipeline
- We use two python scripts to automate the entire data extraction process for this dataset.
- The first script automates the download of CEA *Power Supply Position – Peak* reports from the Monthly Reports Archive. Using Selenium, it navigates the archive interface, selects the required year and month, locates the corresponding PDF report, and downloads it using Requests. Files are saved with a standardized naming convention (`PSP_Peak_YYYY-MM.pdf`) and existing downloads are skipped.
- The second script extracts state-level peak demand data from the downloaded CEA PDF reports using `pdfplumber`. It cleans and standardizes state names, removes aggregate summary rows, handles merged PDF table cells, and converts the extracted information into a structured CSV dataset containing monthly peak demand and peak demand met values for each state.


In [26]:
# Adding project root to sys.path so Python can find the 'src' directory 
sys.path.append(os.path.abspath('..')) 

# Importing python scripts to download pdfs and extract data from the pdfs
from src.cea_downloader import cea_demand_downloader
from src.pdf_extractor import demand_data_extractor

DEMAND_TRENDS_FILEPATH = '../data/raw/cea/cea_csv/cea_monthly_peak_demand.csv'

# Only trigger the extraction scripts if the local raw file doesn't exist
if os.path.isfile(DEMAND_TRENDS_FILEPATH):
    print(f'Verified {DEMAND_TRENDS_FILEPATH} exists. Skipping extraction pipeline.')
else:
    print(f'{DEMAND_TRENDS_FILEPATH} missing, Attempting extraction...')
    cea_demand_downloader()
    demand_data_extractor()

Verified ../data/raw/cea/cea_csv/cea_monthly_peak_demand.csv exists. Skipping extraction pipeline.


### Processing

In [27]:
demand_trends_df = pd.read_csv(DEMAND_TRENDS_FILEPATH)
demand_trends_df

,state,year,month,peak_demand_mw,peak_met_mw,source_file
0,Andhra Pradesh,2015,1,6397.0,6392.0,PSP_Peak_2015-01.pdf
1,Andhra Pradesh,2015,2,6530.0,6518.0,PSP_Peak_2015-02.pdf
2,Andhra Pradesh,2015,3,6790.0,6784.0,PSP_Peak_2015-03.pdf
3,Andhra Pradesh,2015,4,6794.0,6789.0,PSP_Peak_2015-04.pdf
4,Andhra Pradesh,2015,5,6737.0,6732.0,PSP_Peak_2015-05.pdf
...,...,...,...,...,...,...
4879,West Bengal,2026,4,13023.0,12946.0,PSP_Peak_2026-04.pdf
4880,West Bengal,2026,5,13340.0,13340.0,PSP_Peak_2026-05.pdf
4881,West Bengal,2026,6,13570.0,13570.0,PSP_Peak_2026-06.pdf
4882,West Bengal,2026,7,12324.0,12324.0,PSP_Peak_2026-07.pdf


In [28]:
# Standardizing year column
def get_fyear(row):
    year = int(row['year'])
    month = int(row['month'])

    if month >= 4:
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"

demand_trends_df['year'] = demand_trends_df.apply(get_fyear, axis=1)

# Adding month_name column
MONTH_NUM_TO_NAME = {v:k for k,v in MONTH_NAME_TO_NUM.items()}
demand_trends_df['month_name'] = demand_trends_df['month'].map(MONTH_NUM_TO_NAME)

# Standardizing column names
demand_trends_df = demand_trends_df.rename(columns={'year':'fyear', 'month':'month_num'})
demand_trends_df = demand_trends_df[['state', 'fyear', 'month_num', 'month_name', 'peak_demand_mw']]
demand_trends_df

,state,fyear,month_num,month_name,peak_demand_mw
0,Andhra Pradesh,2014-15,1,January,6397.0
1,Andhra Pradesh,2014-15,2,February,6530.0
2,Andhra Pradesh,2014-15,3,March,6790.0
3,Andhra Pradesh,2015-16,4,April,6794.0
4,Andhra Pradesh,2015-16,5,May,6737.0
...,...,...,...,...,...
4879,West Bengal,2026-27,4,April,13023.0
4880,West Bengal,2026-27,5,May,13340.0
4881,West Bengal,2026-27,6,June,13570.0
4882,West Bengal,2026-27,7,July,12324.0


#### Saving as a flat file

In [29]:
PROCESSED_DEMAND_TRENDS_FILEPATH = '../data/processed/flat_files/ENERGY-DEMAND-TRENDS-2015-2026.csv'
if not os.path.isfile(PROCESSED_DEMAND_TRENDS_FILEPATH):
    demand_trends_df.to_csv(PROCESSED_DEMAND_TRENDS_FILEPATH, index=False)

__________________

## Weather Data from NASA POWER API (2015-2026)

### Unified extraction and processing pipeline

In [30]:
WEATHER_DATA_FILEPATH = '../data/raw/nasa_power/SOLAR-DATA.csv'

if os.path.isfile(WEATHER_DATA_FILEPATH):
    print(f'File already exists: {WEATHER_DATA_FILEPATH}, skipping extraction...')
else:
    # Coordinates of centroids for Indian States and UTs
    STATE_COORDINATES = {
        # States
        "Andhra Pradesh": (15.9129, 79.7400),
        "Arunachal Pradesh": (28.2180, 94.7278),
        "Assam": (26.2006, 92.9376),
        "Bihar": (25.0961, 85.3131),
        "Chhattisgarh": (21.2787, 81.8661),
        "Goa": (15.2993, 74.1240),
        "Gujarat": (22.2587, 71.1924),
        "Haryana": (29.0588, 76.0856),
        "Himachal Pradesh": (31.1048, 77.1734),
        "Jharkhand": (23.6102, 85.2799),
        "Karnataka": (15.3173, 75.7139),
        "Kerala": (10.8505, 76.2711),
        "Madhya Pradesh": (22.9734, 78.6569),
        "Maharashtra": (19.7515, 75.7139),
        "Manipur": (24.6637, 93.9063),
        "Meghalaya": (25.4670, 91.3662),
        "Mizoram": (23.1645, 92.9376),
        "Nagaland": (26.1584, 94.5624),
        "Odisha": (20.9517, 85.0985),
        "Punjab": (31.1471, 75.3412),
        "Rajasthan": (27.0238, 74.2179),
        "Sikkim": (27.5330, 88.5122),
        "Tamil Nadu": (11.1271, 78.6569),
        "Telangana": (18.1124, 79.0193),
        "Tripura": (23.9408, 91.9882),
        "Uttar Pradesh": (26.8467, 80.9462),
        "Uttarakhand": (30.0668, 79.0193),
        "West Bengal": (22.9868, 87.8550),

        # Union Territories
        "Andaman and Nicobar Islands": (11.7401, 92.6586),
        "Chandigarh": (30.7333, 76.7794),
        "Dadra and Nagar Haveli and Daman and Diu": (20.1809, 73.0169),
        "Delhi": (28.7041, 77.1025),
        "Jammu and Kashmir": (33.7782, 76.5762),
        "Ladakh": (34.1526, 77.5771),
        "Lakshadweep": (10.5667, 72.6417),
        "Puducherry": (11.9416, 79.8083),
    }


    WEATHER_DATA_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    START_YEAR = 2015
    END_YEAR = 2025

    weather_records = []

    print('Starting NASA Power API extraction...')

    for state, (lat, long) in STATE_COORDINATES.items():
        print(f"Fetching weather data for: {state} ({lat, long})")

        params = {
            "parameters": "ALLSKY_SFC_SW_DWN,T2M",
            "community": "RE",
            "longitude": long,
            "latitude": lat,
            "start": START_YEAR,
            "end": END_YEAR,
            "format": "JSON"
        }

        try:
            response = requests.get(WEATHER_DATA_URL, params=params, timeout = 30)
            response.raise_for_status()
            if response.status_code == 200:
                data = response.json()
                irradiance_dict = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
                temp_dict = data["properties"]["parameter"]["T2M"]

                for yyyymm, irradiance in irradiance_dict.items():
                    #Skipping the 13th month, ie. the yearly average
                    if yyyymm.endswith('13'):
                        continue

                    year = int(yyyymm[:4])
                    month_num = int(yyyymm[4:])
                    temp = temp_dict.get(yyyymm, None)


                    # Deriving financial year to match the fyear format of other datasets
                    if month_num >= 4:
                        fyear = f'{year}-{str(year + 1)[-2:]}'
                    else:
                        fyear = f'{year-1}-{str(year)[-2:]}'
                    

                    weather_records.append({
                        'state': state,
                        'fyear': fyear,
                        'month_num': month_num,
                        'month_name' : MONTH_NUM_TO_NAME[month_num],
                        'temperature_2m_c': temp,
                        'solar_irradiance_kwh_m2_day': irradiance
                    })
            else:
                print(f'Failed to fetch {state}: HTTP {response.status_code}')
                
        except Exception as e:
            print(f'Error fetching {state}: {e}')
        
        #Sleep briefly to be gentle with NASA Power API limits
        time.sleep(0.5)

    weather_df = pd.DataFrame(weather_records)
    os.makedirs(os.path.dirname(WEATHER_DATA_FILEPATH), exist_ok=True)
    weather_df.to_csv(WEATHER_DATA_FILEPATH, index=False)    
    print(f"\nSaved weather & irradiance data from NASA Power to {WEATHER_DATA_FILEPATH}")


File already exists: ../data/raw/nasa_power/SOLAR-DATA.csv, skipping extraction...


In [31]:
weather_df = pd.read_csv(WEATHER_DATA_FILEPATH)

### Saving as a flat file

In [32]:
PROCESSED_WEATHER_DATA_FILEPATH = '../data/processed/flat_files/WEATHER-DATA-2015-2026.csv'
if not os.path.isfile(PROCESSED_WEATHER_DATA_FILEPATH):
    weather_df.to_csv(PROCESSED_WEATHER_DATA_FILEPATH, index=False)